# 01 · What is actually in SEC company facts?

**Question:** before designing any schema, what does this dataset contain, and what in it doesn't make sense?

**Data:** the SEC's `companyfacts` JSON (XBRL numbers pulled from every 10-K / 10-Q a company has filed), one file per company, downloaded with `python -m edgar_bitemporal.fetch`.

No database in this notebook: just the raw files and pandas, so the observations that drive the schema are visible first.

In [1]:
import os, json, pathlib
import pandas as pd
from edgar_bitemporal.flatten import flatten_companyfacts

pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30)

RAW = pathlib.Path(os.environ.get("EDGAR_RAW_DIR", "../data/raw"))
rows, names = [], {}
for f in sorted(RAW.glob("CIK*.json")):
    payload = json.loads(f.read_text())
    names[int(payload["cik"])] = payload["entityName"]
    rows.extend(flatten_companyfacts(payload, tags=None, taxonomies=("us-gaap", "dei")))

df = pd.DataFrame(rows)
for c in ("period_start", "period_end", "filed"):
    df[c] = pd.to_datetime(df[c])
df["company"] = df.cik.map(names)
print(f"{len(names)} companies, {len(df):,} reported facts")

1 companies, 34,391 reported facts


## 1 · The shape of it

In [2]:
df.groupby(["company", "taxonomy"]).agg(
    facts=("value", "size"), tags=("tag", "nunique"),
    first_filed=("filed", "min"), last_filed=("filed", "max"))

facts  tags first_filed last_filed
company            taxonomy                                    
THE BOEING COMPANY dei          87     2  2009-07-22 2026-07-28
                   us-gaap   34304   742  2009-07-22 2026-07-28

In [3]:
df.form.value_counts()

form
10-Q    21206
10-K    13185
Name: count, dtype: int64

## 2 · One period, reported many times

Every 10-Q and 10-K repeats prior periods as comparatives. So a "fact" (company, tag, unit, period) shows up once **per filing that mentions it**. How often, and does the number ever change?

In [4]:
KEY = ["cik", "taxonomy", "tag", "unit", "period_start", "period_end"]
keys = (df.groupby(KEY, dropna=False)
          .agg(n_reports=("accn", "nunique"), n_values=("value", "nunique"))
          .reset_index())

print(f"{len(keys):,} distinct periods")
print(f"{(keys.n_reports > 1).mean():.1%} are reported in more than one filing")
print(f"{(keys.n_values > 1).mean():.1%} were reported with more than one value (restated)")
keys.n_reports.describe()

17,533 distinct periods
60.0% are reported in more than one filing
3.1% were reported with more than one value (restated)


count    17533.000000
mean         1.961501
std          1.181243
min          1.000000
25%          1.000000
50%          2.000000
75%          2.000000
max         13.000000
Name: n_reports, dtype: float64

Pick the most-revised balance-sheet or income number and look at its full history, one row per filing:

In [5]:
focus = keys[(keys.n_values > 1) & keys.tag.isin(["Assets", "StockholdersEquity", "NetIncomeLoss"])]
top = focus.sort_values(["n_values", "n_reports"], ascending=False).head(1)
history = (df.merge(top[KEY], on=KEY, how="inner")
             .sort_values(["filed", "accn"])[["company", "tag", "period_end", "filed", "form", "accn", "value"]])
history

,company,tag,period_end,filed,form,accn,value
0,THE BOEING COMPANY,Assets,2014-12-31,2015-02-12,10-K,0000012927-15-000011,9.919800e+10
1,THE BOEING COMPANY,Assets,2014-12-31,2015-04-22,10-Q,0000012927-15-000019,9.919800e+10
2,THE BOEING COMPANY,Assets,2014-12-31,2015-07-22,10-Q,0000012927-15-000055,9.919800e+10
3,THE BOEING COMPANY,Assets,2014-12-31,2015-10-21,10-Q,0000012927-15-000068,9.919800e+10
4,THE BOEING COMPANY,Assets,2014-12-31,2016-02-10,10-K,0000012927-16-000099,9.292100e+10


## 3 · What do `fy` and `fp` mean?

The JSON has `fy` (fiscal year) and `fp` (fiscal period) on every fact, which looks like a label for the period the number covers. Test that on annual figures from 10-Ks:

In [6]:
days = (df.period_end - df.period_start).dt.days
annual = df[(df.form == "10-K") & days.between(350, 380)].copy()
annual["fy_minus_period_year"] = annual.fiscal_year - annual.period_end.dt.year
annual.fy_minus_period_year.value_counts().sort_index().rename("facts")

fy_minus_period_year
-1       1
 0    2637
 1    2463
 2    2278
 3       2
 4       2
Name: facts, dtype: int64

If `fy` described the period, every row would be `0`. It doesn't: a 10-K re-reports prior years as comparatives and stamps them all with **its own** fiscal year. So `fy`/`fp` identify the *filing*, not the period; using them to pick "the FY2016 number" silently mixes vintages. The model below keys on the period's own dates and the filing's accession number instead.

## 4 · Instants vs durations, and cumulative periods

Balance-sheet items are point-in-time (no `start`). Flow items have a start and an end, but not all durations are quarters:

In [7]:
print(f"{df.period_start.isna().mean():.1%} of facts are instants (no start date)")
bucket = pd.cut(days, [-1, 31, 100, 200, 290, 380, 10**6],
                labels=["<= 1 month", "~ quarter", "~ half-year (YTD)", "~ 9 months (YTD)", "~ year", "longer"])
bucket.value_counts().sort_index().rename("duration facts")

38.1% of facts are instants (no start date)


<= 1 month            107
~ quarter            7177
~ half-year (YTD)    3371
~ 9 months (YTD)     3210
~ year               7435
longer                  0
Name: duration facts, dtype: int64

10-Qs report year-to-date totals alongside the single quarter, so a naive pivot of "all Q2 values" double counts. Duration length has to be part of the key.

## 5 · The same concept under different tag names

Companies switch XBRL tags over time (new accounting standards, taxonomy updates). Revenue is the classic case:

In [8]:
CANDIDATES = ["Revenues", "SalesRevenueNet", "SalesRevenueGoodsNet",
              "RevenueFromContractWithCustomerExcludingAssessedTax"]
(annual[annual.tag.isin(CANDIDATES)]
   .groupby(["company", "tag"])
   .agg(first_period=("period_end", "min"), last_period=("period_end", "max"), facts=("value", "size")))

first_period last_period  facts
company            tag                                                                               
THE BOEING COMPANY RevenueFromContractWithCustomerExcludingAssesse...   2017-12-31  2019-12-31      3
                   Revenues                                             2007-12-31  2025-12-31     51
                   SalesRevenueGoodsNet                                 2007-12-31  2017-12-31     27

A time series of "revenue" has to be stitched from several tags. That's a modelling decision to make explicitly, not something to discover in a backtest.

## 6 · Things that look wrong

**Filed before the period ended.** Filings normally report periods that have already finished, so any that don't deserve a look:

In [9]:
early = df[df.filed < df.period_end]
print(f"{len(early)} facts filed before their period end")
early[["company", "taxonomy", "tag", "period_end", "filed", "form", "value"]].head(10)

3 facts filed before their period end


,company,taxonomy,tag,period_end,filed,form,value
10163,THE BOEING COMPANY,us-gaap,EffectiveIncomeTaxRateReconciliationAtFederalS...,2018-12-31,2018-02-12,10-K,0.21
10164,THE BOEING COMPANY,us-gaap,EffectiveIncomeTaxRateReconciliationAtFederalS...,2018-12-31,2018-04-25,10-Q,0.21
10165,THE BOEING COMPANY,us-gaap,EffectiveIncomeTaxRateReconciliationAtFederalS...,2018-12-31,2018-07-25,10-Q,0.21


In [10]:
early.groupby("tag").size().rename("facts filed before period end")

tag
EffectiveIncomeTaxRateReconciliationAtFederalStatutoryIncomeTaxRate    3
Name: facts filed before period end, dtype: int64

Check which tags these are. Forward-looking disclosures (a tax rate that applies to the current year, say) can legitimately describe a period that hasn't ended yet, so this is a flag to review, not proof of a bad row. That is why it is a monitored check rather than a table constraint.

**Sign flips.** The same period reported with the opposite sign by a later filing:

In [11]:
d = df.sort_values(KEY + ["filed", "accn"])
d["prev_value"] = d.groupby(KEY, dropna=False)["value"].shift()
flips = d[d.value * d.prev_value < 0]
print(f"{len(flips)} re-reports flipped the sign of the previous value")
flips.groupby(["company", "tag"]).size().sort_values(ascending=False).head(10).rename("sign flips")

148 re-reports flipped the sign of the previous value


company             tag                                                                                    
THE BOEING COMPANY  AccrualForEnvironmentalLossContingenciesIncreaseDecreaseForRevisionInEstimates             11
                    OtherComprehensiveIncomeUnrealizedHoldingGainLossOnSecuritiesArisingDuringPeriodTax        11
                    CostOfRevenue                                                                              10
                    OtherComprehensiveIncomeLossTax                                                             8
                    OtherComprehensiveIncomeLossCashFlowHedgeGainLossReclassificationAfterTax                   7
                    OtherComprehensiveIncomeLossCashFlowHedgeGainLossBeforeReclassificationTax                  7
                    EffectiveIncomeTaxRateReconciliationTaxCreditsResearch                                      6
                    OtherComprehensiveIncomeUnrealizedGainLossOnDerivativesArisingDuringPeriod

In [12]:
flips[["company", "tag", "period_end", "filed", "form", "prev_value", "value"]].head(6)

,company,tag,period_end,filed,form,prev_value,value
545,THE BOEING COMPANY,AccrualForEnvironmentalLossContingenciesIncrea...,2010-09-30,2011-10-26,10-Q,-88000000.0,88000000.0
547,THE BOEING COMPANY,AccrualForEnvironmentalLossContingenciesIncrea...,2010-12-31,2012-02-09,10-K,-108000000.0,108000000.0
549,THE BOEING COMPANY,AccrualForEnvironmentalLossContingenciesIncrea...,2011-06-30,2012-07-25,10-Q,-99000000.0,99000000.0
557,THE BOEING COMPANY,AccrualForEnvironmentalLossContingenciesIncrea...,2012-06-30,2013-07-24,10-Q,45000000.0,-45000000.0
559,THE BOEING COMPANY,AccrualForEnvironmentalLossContingenciesIncrea...,2012-09-30,2013-10-23,10-Q,65000000.0,-65000000.0
561,THE BOEING COMPANY,AccrualForEnvironmentalLossContingenciesIncrea...,2012-12-31,2014-02-14,10-K,73000000.0,-73000000.0


## 7 · Where does the history start?

`companyfacts` is built from XBRL filings, which began around 2009. A period that ended before that can only appear as a *comparative* inside a later filing. For those periods, how long after the period ended did the number first appear?

In [13]:
first = df.groupby(KEY, dropna=False).filed.min().reset_index(name="first_filed")
first["lag_days"] = (first.first_filed - first.period_end).dt.days
late = first[first.lag_days > 400]
print(f"earliest filing in the data: {df.filed.min():%Y-%m-%d}")
print(f"{len(late):,} of {len(first):,} periods ({len(late) / len(first):.1%}) first appear more than 400 days after they ended")
print(f"latest period end among them: {late.period_end.max():%Y-%m-%d}")
first.lag_days.describe(percentiles=[.5, .9, .99]).rename("days from period end to first appearance").to_frame().T

earliest filing in the data: 2009-07-22
665 of 17,533 periods (3.8%) first appear more than 400 days after they ended
latest period end among them: 2023-12-31


,count,mean,std,min,50%,90%,99%,max
days from period end to first appearance,17533.0,69.646096,136.377498,-322.0,28.0,45.0,770.0,1932.0


For those periods, `filed` is when the number first appears in the XBRL record, **not** when the original 10-K was published. As-of queries before a company's first XBRL filing are unreliable, so the lag and backtest analysis in notebook 03 only uses periods first reported within 150 days of period end.

## What this means for the design

1. **Keep every report.** Restatements are real (section 2), so overwriting loses history. Each report of a fact is its own immutable row.
2. **Two time axes.** *Valid time* is the period (`period_start`, `period_end`). *Transaction time* is the filing date `filed`: when the market could first see the number.
3. **Key on the period and the filing, never on `fy`/`fp`** (section 3).
4. **Duration is part of the key** (section 4).
5. **Don't reject odd rows at load.** Some of the oddities in section 6 are legitimate (forward-looking disclosures) and others are conventions that changed between filings, so rejecting them would delete real history. They are loaded, and counted by `sql/checks.sql`.
6. **Transaction time is "first seen in the XBRL record".** For pre-XBRL history that is later than real publication (section 7), so analyses must exclude it.

Next: [02 · The bitemporal model](02_bitemporal_model.ipynb).